<a href="https://colab.research.google.com/github/ajzal4you/Master-Project-/blob/main/XAI_GradCAM_Fast_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XAI Grad-CAM Fast Starter (Colab)
**Created:** 2025-08-13 15:22 UTC  
This notebook generates Grad‑CAM overlays for your trained classification models (Brain/Breast/Kidney/Bone).  
Just edit the **CONFIG** cell and run top-to-bottom.

**What you get**
- Drive mount (optional)
- Model load (ResNet18 by default; custom CNN template included)
- Grad‑CAM on a single image or a folder
- Saved overlays in `/MSc_Project/xai_outputs/<module>/` on Drive


## 1) Install dependencies (Colab)

In [ ]:
#@title Install (quiet)
import sys, subprocess, importlib

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["opencv-python", "pillow", "matplotlib", "torchvision", "captum"]:
    try:
        importlib.import_module(pkg.split("==")[0].split(">=")[0])
    except Exception:
        pip_install(pkg)

print("✅ Installs checked.")

✅ Installs checked.


## 2) Configure paths and options

In [ ]:
#@title CONFIG — EDIT THESE
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

# Paths (edit to your Drive layout if needed)
PROJECT_ROOT = "/content/drive/MyDrive/MSc_Project"  #@param {type:"string"}
MODULE_NAME  = "brain_classification"                #@param {type:"string"}

# Model & data
MODEL_PATH   = f"{PROJECT_ROOT}/{MODULE_NAME}/models/brain_model_resnet18.pth"  #@param {type:"string"}
CLASS_NAMES  = ["glioma","meningioma","no_tumor","pituitary"]  #@param {type:"raw"}
NUM_CLASSES  = len(CLASS_NAMES)
IMG_SIZE     = 224  #@param {type:"number"}
INPUT_IS_GRAY = False  #@param {type:"boolean"}

# Single image test (optional)
SAMPLE_IMG   = f"{PROJECT_ROOT}/{MODULE_NAME}/samples/sample_001.png"  #@param {type:"string"}

# Batch folder (optional)
INPUT_DIR    = f"{PROJECT_ROOT}/{MODULE_NAME}/samples"  #@param {type:"string"}

# Output directory
OUT_DIR      = f"{PROJECT_ROOT}/xai_outputs/{MODULE_NAME}"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("✅ CONFIG ready.")

✅ CONFIG ready.


## 3) Mount Google Drive (if enabled)

In [ ]:
#@title Mount Drive
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
    except ValueError as e:
        if "Mountpoint must not already contain files" in str(e):
             print("Attempting to unmount and remount...")
             drive.flush_and_unmount()
             drive.mount('/content/drive', force_remount=True)
        else:
            raise e
print("✅ Drive status OK.")

Attempting to unmount and remount...
Drive not mounted, so nothing to flush and unmount.


ValueError: Mountpoint must not already contain files

## 4) Imports

In [ ]:
#@title Imports
import glob
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision import models

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 5) Load model
Choose ResNet18 (default) or uncomment the Custom CNN block and comment out ResNet.

In [ ]:
#@title Load model (ResNet18 by default)
# ---- Option A: ResNet18 (default) ----
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# ---- Option B: Custom CNN (uncomment to use) ----
# class BrainTumorCNN(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
#         self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
#         self.pool  = nn.MaxPool2d(2, 2)
#         self.fc1   = nn.Linear(32*56*56, 128)
#         self.fc2   = nn.Linear(128, num_classes)
#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = x.view(-1, 32*56*56)
#         x = F.relu(self.fc1(x))
#         return self.fc2(x)
# model = BrainTumorCNN(NUM_CLASSES)

# Load weights
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model = model.to(device).eval()
print("✅ Model loaded:", type(model).__name__)

## 6) Preprocessing (must match training)

In [ ]:
#@title Define transforms
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    # EDIT if you used custom normalization during training
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])
print("✅ Transforms ready.")

## 7) Helper functions and Grad‑CAM class

In [ ]:
#@title Helpers + Grad-CAM
from typing import Optional, Tuple

def ensure_rgb(img: Image.Image) -> Image.Image:
    """If grayscale and INPUT_IS_GRAY==True, repeat channels to 3."""
    if img.mode == 'RGB':
        return img
    if INPUT_IS_GRAY:
        arr = np.array(img)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        elif arr.ndim == 3 and arr.shape[2] == 1:
            arr = np.repeat(arr, 3, axis=2)
        return Image.fromarray(arr.astype(np.uint8))
    return img.convert('RGB')

class GradCAM:
    def __init__(self, model: nn.Module, target_layer: Optional[nn.Module]=None):
        self.model = model.eval()
        self.device = next(model.parameters()).device
        self.target_layer = target_layer or self._find_last_conv(model)
        self.activations = None
        self.gradients = None
        self.target_layer.register_forward_hook(self._fwd_hook)
        self.target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def _find_last_conv(self, model: nn.Module) -> nn.Module:
        last = None
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                last = m
        if last is None:
            raise RuntimeError("No Conv2d found; Grad-CAM requires conv layers.")
        return last

    def generate(self, x: torch.Tensor, class_idx: Optional[int]=None) -> Tuple[np.ndarray, int]:
        x = x.to(self.device)
        self.model.zero_grad()
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(torch.argmax(logits, dim=1).item())
        score = logits[:, class_idx]
        score.backward(retain_graph=True)
        grads = self.gradients  # [B,C,H,W]
        acts  = self.activations
        weights = grads.mean(dim=(2,3), keepdim=True)  # [B,C,1,1]
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().detach().cpu().numpy()
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, class_idx

def overlay_cam_on_image(img_rgb: np.ndarray, cam: np.ndarray, alpha: float=0.40) -> np.ndarray:
    heat = (cam * 255).astype(np.uint8)
    heat = cv2.applyColorMap(heat, cv2.COLORMAP_JET)[:, :, ::-1]  # BGR->RGB
    ov = (alpha * heat + (1 - alpha) * img_rgb).astype(np.uint8)
    return ov

print("✅ Grad-CAM ready.")

## 8) Run on a single image

In [ ]:
#@title Single image demo
from pathlib import Path

def run_one_image(img_path: str) -> str:
    raw = Image.open(img_path)
    raw = ensure_rgb(raw)
    raw_resized = raw.resize((IMG_SIZE, IMG_SIZE))
    x = transform(raw).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs)); pred_conf = float(probs[pred_idx])
    cam_engine = GradCAM(model)
    cam, _ = cam_engine.generate(x, class_idx=pred_idx)
    overlay = overlay_cam_on_image(np.array(raw_resized), cam, alpha=0.40)
    name = os.path.splitext(os.path.basename(img_path))[0]
    out = os.path.join(OUT_DIR, f"{name}_overlay_pred{pred_idx}_{pred_conf:.2f}.png")
    Image.fromarray(overlay).save(out)
    print(f"Saved: {out} | pred={CLASS_NAMES[pred_idx] if pred_idx < len(CLASS_NAMES) else pred_idx} ({pred_conf:.2f})")
    # Show quick preview
    plt.figure(figsize=(10,4))
    plt.subplot(1,3,1); plt.title("Input"); plt.axis("off"); plt.imshow(np.array(raw_resized))
    plt.subplot(1,3,2); plt.title("Grad-CAM"); plt.axis("off"); plt.imshow(cam, cmap='jet')
    plt.subplot(1,3,3); plt.title("Overlay"); plt.axis("off"); plt.imshow(overlay)
    plt.show()
    return out

if os.path.isfile(SAMPLE_IMG):
    _ = run_one_image(SAMPLE_IMG)
else:
    print("[INFO] SAMPLE_IMG not found:", SAMPLE_IMG)

## 9) Batch over a folder (optional)

In [ ]:
#@title Batch run
def run_folder(input_dir: str) -> None:
    paths = sorted(glob.glob(os.path.join(input_dir, "*.png")) +
                   glob.glob(os.path.join(input_dir, "*.jpg")) +
                   glob.glob(os.path.join(input_dir, "*.jpeg")))
    print(f"Found {len(paths)} images in {input_dir}")
    for p in paths:
        try:
            run_one_image(p)
        except Exception as e:
            print(f"[WARN] Failed {p}: {e}")

if os.path.isdir(INPUT_DIR):
    run_folder(INPUT_DIR)
else:
    print("[INFO] INPUT_DIR not found:", INPUT_DIR)

## 10) Next steps
- Duplicate this notebook for **breast/kidney/bone** modules by changing only `MODULE_NAME`, `MODEL_PATH`, `CLASS_NAMES`, `IMG_SIZE`.
- If your data is single-channel (X-ray/CT) set `INPUT_IS_GRAY=True`.
- For deeper analysis later, add **Captum Integrated Gradients** or **SHAP**.
